# 4b · MIL training — instance-level, the instrument for Q2

> **The criterion in §2 was fixed on 13.08.2026, before any model existed, and nothing here has been
> run.** That ordering is the point: a structured-looking result read after the fact is not evidence
> ([TODO](../docs/TODO.md), *Agreed plan, Step 2*). §2 is now closed — every bar in it is a permutation
> null, a comparison against `4a`, or a collapse test, and no magnitude in it was chosen by judgement.
> The cells below implement it and may not be extended with a stage that is not in it.

## What this notebook is for

**Q2 — does within-line transcriptional heterogeneity survive into a model's per-cell response
predictions?** Scoped this way by Selin, 13.08.2026.

Concretely: do cells of one cell line differ enough in their representation — `pca` or `scgpt` — for a
model to assign them different response values, reproducibly and not as an artifact? Kinker et al.
(2020) documented recurrent heterogeneity programs in this dataset, but whether that heterogeneity
survives *our* preprocessing and embedding is **not assumed** — stage 0 measures it, and stage 0 can
fail.

It is the question the project is built around, because relapse is driven by rare surviving
subpopulations rather than by the average cell.

It is **structurally unanswerable** under `4a_percell_training`'s per-cell model: every cell of a line
carries that line's label, so the objective penalises exactly the within-line variation Q2 asks about
([Step 03](../docs/steps/03-model-and-training-design.md#every-cell-of-a-line-carries-the-identical-label)).
MIL is the smallest change that makes the question askable: a **bag of cells → one line label**
constrains only the aggregate, leaving the model free to differ between cells of the same line.

**The representation is part of the question, not a setting (Selin, 13.08.2026).** Q2 is asked of
`pca` and of `scgpt` separately and both arms run. One representation carrying within-line structure
while the other does not is a **result**, not a nuisance — it locates the structure in the encoding
rather than in the model.

### What Q2 claims here, and what it does not

Three questions had been running together under one name. Separated 13.08.2026:

| | | reachable with SCP542 + CTRPv2 |
|---|---|---|
| **a** | do per-cell predictions vary within a line, reproducibly, and not as a sequencing artifact? | **yes** — stages 0, 1, 2 and 6 below |
| **b** | is that variation *real* cellular heterogeneity of drug response? | **no** |
| **c** | does it predict *which* cells survive treatment? | **no** |

**(b) and (c) are out of reach for want of measurements, not because of this design.** No per-cell
response was ever measured — every label is one number per (cell line, drug) — and none of the four
primary sources carries post-treatment single-cell data. No threshold and no architecture closes that
gap; it takes a different dataset.

**So this notebook answers (a), and only (a).** That is worth having on its own terms: (a) is a
**necessary condition** for (b), so a model failing (a) has definitively not learned heterogeneity.
§4's program analysis is a **hint toward** (b) and explicitly not evidence for it. The write-up states
(b) and (c) as limitations of the data rather than leaving the narrowing implicit.

⚠️ **Candidate routes to (b) are recorded in
[Step 01](../docs/steps/01-datasets-and-harmonization.md#post-treatment-single-cell-data--what-would-be-needed-for-q2-b-and-what-exists)**
— MIX-seq, the palbociclib file already on disk, and lineage-barcoding studies, each with what it can
and cannot establish. **None is scoped, and none belongs to this notebook**; they are written down so
they are not rediscovered here as shortcuts.

## Decisions already taken

**Instance-level, not attention pooling (Selin, 12.08.2026).** Every cell gets its own *predicted
response*, and the line prediction aggregates them — rather than every cell getting an attention
*weight* over a pooled embedding. The two are the standard MIL alternatives (Ilse, Tomczak & Welling,
*Attention-based Deep Multiple Instance Learning*, ICML 2018): embedding-level usually predicts better,
instance-level is readable at the level of the individual instance. Q2 is a question about readability,
so the trade is taken deliberately and the cost in predictive performance is expected.

⚠️ **One consequence, recorded so it is not rediscovered:** selecting "the top-k cells" *by their
predicted value* and scoring that subset against the line's true response is biased by construction —
the extremes are shifted away from the line mean because they were chosen for being extreme. The
subpopulation-predictivity test that would have used it is therefore **not** in the criterion below.
It becomes available only if an attention weight is added alongside the per-cell predictions.

**Mean pooling, full-line bags, and the aggregator is never revisited (Selin, 13.08.2026).** The bag
prediction is the mean of its cells' predicted responses, over **all** of that line's cells. It matches
the synthetic control's own construction — a mixture-weighted *average* label — and it is fixed now,
before the model exists.

**Bag size is not a batching parameter.** For a sub-bag of `B` cells from a line of `n`, the expected
loss is `(p̄_n − y)² + (σ²/B)·(1 − (B−1)/(n−1))`: at `B = 1` that is exactly `4a`'s loss, at `B = n` the
variance term vanishes. Bag size therefore dials continuously between the per-cell model and MIL, and
sets how hard the objective charges for the very quantity Q2 measures. Full-line bags delete the term
outright — which is what stage 1's comparison against `4a` tests — and they are also what keeps the
[depth-weighting defect](../docs/steps/03-model-and-training-design.md#-open-defect--the-loss-weights-cell-lines-by-how-deeply-they-were-sequenced)
resolved, since under fixed-size sub-bags a deeply sequenced line yields proportionally more bags and
the depth weighting returns. Cost: one gradient step per line, and memory scaling with the largest
line.

⚠️ **Its cost, taken deliberately.** Mean pooling under an MSE-family loss carries a real shrinkage
incentive: collapsing every cell onto the bag mean is a minimiser whenever the model cannot do better,
and nothing in the objective rewards spreading cells apart. Both alternatives were considered and
rejected — a **variance term in the loss** would make stage 1 pass by construction and would change the
loss, which [the governing rule](../docs/TODO.md) forbids; a **max or top-quantile aggregator** does not
force variation either (the max of equal per-cell values is that value) and mismatches an averaged
label.

**The retry is ruled out, and that is the part that binds.** If stage 7 fails, the aggregator is *not*
swapped for one that passes. *Fail → change the instrument → retry* is a forking path moved down one
level, and it would leave any subsequent positive unattributable. A stage-7 failure **ends the run**,
and what it means is fixed here rather than after the numbers are seen: **Q2 unanswered, the instrument
was not demonstrated** — never "no heterogeneity found".

**Same scorer as the per-cell model.** This notebook writes out-of-fold predictions in the shared
format — one row per cell line × drug × arm — so [`5_evaluation`](5_evaluation.ipynb) computes order,
top-of-order, values and spread for MIL and the per-cell model through identical code. The two are
comparable because they went through the same scorer, not because two notebooks agree by convention.

**Loss:** whatever `4a_percell_training` settles on, unchanged, so the architecture is the only thing
that moves ([the governing rule](../docs/TODO.md)). Ranking losses (RankNet, LambdaRank) become
well-posed *here* and nowhere earlier — they need one score per cell line, which is what a bag produces
— but they are a second change and belong to a later run, not this one.


## 2 · What counts as a positive Q2 result — fixed before the run

**Settled by Selin, 13.08.2026. Every bar below is a permutation null, a comparison against
`4a_percell_training`, or a collapse test — the criterion contains no magnitude chosen by judgement.**

That was not true of the first draft, which carried three invented numbers: a floor on stage 0, an
AUROC bar on stage 7, and a fraction on stage 2. Each was removed by **replacing the bar with a null or
a comparison**, not by picking a better number. The one that mattered was stage 7's: a stage-7 failure
ends the run, so an arbitrary bar there decided whether the project reported anything at all.

**There is no ground truth for within-line heterogeneity of drug response.** Every label is one number
per (cell line, drug); no per-cell response was ever measured, and SCP542 carries no post-treatment
single-cell data, so the ideal test — do the model's resistant cells match the cells that actually
survive treatment — cannot be run here. That is question **(b)** in §1, out of reach for want of
measurements.

What can be established is narrower and still worth having: **that the model's per-cell predictions
vary within a line, that the variation is reproducible rather than noise, and that it is not a
sequencing artifact.**

Five stages, in this order. Two are preconditions, one is a necessary condition, one is the test, one
is a veto.

| # | Stage | Role | Passes when |
|---|---|---|---|
| **0** | **Input ceiling** — within-line dispersion of the cell representations, per representation, **before any training** | precondition on the **input** | the cells of a line are not collapsed to a point (§2.1) |
| **7** | **Synthetic positive control** — bags mixed from two lines of known, slightly different response; scored **inside the bag**, source-A cells against source-B cells | precondition on the **instrument** | the within-bag rank separation beats its permutation null (§2.2) |
| **1** | **Spread** — within-line standard deviation of per-cell predicted responses | necessary condition | it exceeds `4a`'s, on the same lines and drugs (§2.3) |
| **2** | **Reproducibility** — do independent seeds assign high and low predictions to the *same* cells? | **the test** | per-cell cross-seed agreement beats the shuffled-cell null (§2.4) |
| **6** | **Confound regression** — per-cell predictions against total counts, genes detected, mitochondrial fraction and cell-cycle score | **veto** | the confounds do *not* explain the variation (§2.5) |

**Why stage 0 comes first.** It costs no training. If the cells of a line collapse to a point in the
representation, no model of any kind can assign them different values, and every later stage is
measuring the wrong thing. It also separates two findings that a training run alone conflates: *the
input carries no within-line structure* and *the model did not use the structure that was there* — the
first is a statement about the representation, the second about the model. Because Q2 is asked of `pca`
and `scgpt` separately, stage 0 is per-representation and **may pass for one and fail for the other**,
which is itself a result.

**Why stage 7 comes before the rest.** Without it a negative is uninterpretable — "no heterogeneity
found" cannot be distinguished from "this method cannot find heterogeneity". With it, a negative becomes
a result: no detectable heterogeneity, by an instrument shown to detect it when present, **at a measured
sensitivity that is reported alongside the negative.**

**Why stage 7 is scored inside the bag and not on the bag (Selin, 13.08.2026).** The bag prediction is
the mean of its cells' predictions, so a model that assigns **every cell in the bag an identical value**
can land the bag mean exactly right and pass a bag-level test — while doing none of what stages 1, 2
and 6 go on to measure. Bag-level recovery would certify a shrunk instrument. It is still computed
(predicted bag value against mixture weight, over weights 0 … 1 in steps of 0.25) and **reported as a
diagnostic; it cannot pass the stage.**

**Why stage 6 is a veto and not an analysis.** It looks descriptive, but it can turn a pass into a
fail: predictions that replicate across seeds *and* are explained by library size are a sequencing
artifact, not biology. Pre-registered here so it cannot become something run only when the answer is
unwelcome.

### 2.1 · Stage 0 — the input ceiling, and why it has no floor

**Statistic:** per representation, the **within-line share of total variance** — mean over lines of the
within-line variance of the cell embeddings, divided by the total variance over all cells. It is one
minus the intraclass correlation, an ordinary quantity, and being a ratio it is scale-free, so `pca`
and `scgpt` are on one axis despite sharing no units.

**It is reported, and it fails only on collapse.** An earlier draft put a floor at 0.10. That number
was dropped on 13.08.2026 (Selin) for two reasons: it had no source, and it was *stricter than the
precondition it stood for*. Stage 0 exists to rule out one thing — that the cells of a line are
numerically indistinguishable, so that no model of any kind could separate them. That is the bar.
Anything above it is a matter of degree and belongs in the report as a number, not in a gate.

⚠️ **Timing.** The embeddings on disk predate the preprocessing corrections and R1 re-embeds, so a
number computed before R1 is indicative, not final. The *shape* of the answer — spread versus collapsed
— is unlikely to flip.

### 2.2 · Stage 7 — the synthetic positive control

**Construction.** For each panel drug, take cell-line pairs whose measured responses differ by an amount
in the **bottom quartile** of that drug's pairwise `|y_A − y_B|`, mix their cells at weight **0.5**, and
label the bag with the mixture-weighted response. Inside such a bag the per-cell ground truth is known
by construction: A-cells should score near `y_A`, B-cells near `y_B`.

**Why the pairs are chosen to be close, not far apart (Selin, 13.08.2026).** The only response
differences this project ever measured are *between* cell lines, so any manufactured control inherits a
between-line magnitude — which is larger than any plausible within-line heterogeneity. Pairing on a
large gap gives a control that is easy to pass and useless as evidence about the regime Q2 actually
operates in. The bottom quartile brings the planted difference down toward that regime. Its cost, taken
deliberately: the control is harder to pass, and a stage-7 failure ends the run — so the run can be
ended by a control tuned finer than the labels can support.

**Quantity: within-bag rank separation, measured as AUROC.** Take "this cell came from line A" as the
group label and the model's predicted response as the score. AUROC is then the probability that a
randomly drawn A-cell is predicted higher than a randomly drawn B-cell — 0.5 is no separation, 1.0 is
perfect. Nothing is being classified; this is a rank statistic, and it is exactly the **Mann–Whitney U**
statistic divided by `n_A · n_B`.

**Pass condition: it beats its permutation null** — shuffle which cells came from which source line,
within the bag, and recompute. That null *is* the Mann–Whitney null, so stage 7 is an ordinary Wilcoxon
rank-sum test and needs no threshold chosen by judgement. **The AUROC itself is reported as the
instrument's measured sensitivity**, and every negative result downstream is stated with it attached.

**Why ordering rather than recovered magnitude (Selin, 13.08.2026).** The alternative was the recovered
gap fraction, `(mean prediction on A-cells − mean prediction on B-cells) / (y_A − y_B)`, which is on the
label's own scale and directly interpretable. It was rejected on two grounds. It is
**calibration-sensitive**, and mean pooling gives the model a standing shrinkage incentive, so a model
that orders cells correctly but pulls them toward the bag mean scores low — meaning stage 7 could fail
for the same reason stage 1 would, which costs stage 7 its independence from the stage it licenses. And
under bottom-quartile pairing its denominator `y_A − y_B` is small by construction, so the ratio is
noisy exactly where it is being used. AUROC is immune to shrinkage because it reads only order, which is
what stages 1 and 2 rest on.

The cost, stated: AUROC carries no scale. It says the model separated the groups, not by how much. The
recovered gap fraction is therefore still **computed and reported** beside it — as a description, not a
gate.

### 2.3 · Stage 1 — spread, against `4a` rather than against a number

**Pass condition: MIL's within-line standard deviation of per-cell predictions exceeds `4a`'s**, on the
same cell lines, the same drugs, the same folds and the same representation. No margin, no fraction.

**Why no threshold is needed.** For one line with per-cell predictions `p_c` and its single label `y`:

```
(1/n) Σ_c (p_c − y)²   =   (p̄ − y)²   +   Var_c(p)
   4a's loss, regrouped     MIL's loss     the term MIL deletes
```

an identity, not an approximation
([Step 03](../docs/steps/03-model-and-training-design.md#the-penalty-on-within-line-variation-is-exact-not-figurative-13082026)).
`4a`'s objective charges for within-line variance at full weight in every batch; MIL's mean-pooled
objective does not contain the term at all. So `4a`'s within-line spread is spread that survived an
explicit penalty, and MIL's is spread with that penalty removed. **Comparing the two tests precisely the
deleted term** — which is why an invented margin would add nothing except a number to defend.

⚠️ **This makes `4a` a dependency of `4b`.** Stage 1 cannot be computed until `4a` has run and written
its per-cell predictions, on matching folds, target and representation. That ordering is now a hard edge
in the R-sequence and did not exist before 13.08.2026.

⚠️ **Do not read `4a`'s existing `pred_std` for this.** The column of that name in
`outputs/panel/panel_per_drug_correlation.csv` is the spread of *line-level* predictions **across** cell
lines — a between-line quantity, and the opposite of what stage 1 needs.

### 2.4 · Stage 2 — reproducibility, the actual test

**Pass condition: per-cell agreement across independent seeds beats the shuffled-cell null** — the same
agreement statistic recomputed after permuting cell identities within each line, which destroys any
cell-specific signal while preserving the marginal distribution of predictions. **Three seeds**
(13.08.2026), which is the minimum that makes "the same cells, under different initializations" a
comparison; more seeds only sharpen the estimate and change no bar.

**The last invented number was removed here too.** An earlier draft required the agreement to reach half
of what stage 7 produced. That fraction was dropped on 13.08.2026 (Selin) on the same grounds as the
others: the shuffled-cell null already separates signal from noise, and a magnitude on top of it would
have been chosen rather than derived. The agreement value is **reported**, so a result that is
statistically clear but small is visible as such rather than hidden behind a pass.

### 2.5 · Stage 6 — the confound veto

Regress each cell's predicted response on total counts, genes detected, mitochondrial fraction and
cell-cycle score, **within line**. The veto fires when the confounds explain the within-line variation —
in which case stages 1 and 2 have measured a sequencing artifact that happens to reproduce across seeds,
because the confounds themselves reproduce across seeds.

### 2.6 · Run scope

| | | |
|---|---|---|
| **Representations** | `pca` and `scgpt`, both, separately | Q2 is asked of each; a split answer locates the structure in the encoding |
| **Loss / weighting** | `alpha = 0.5` only, `4a`'s default | the architecture is the change under test; sweeping `alpha` as well would make a difference unattributable ([the governing rule](../docs/TODO.md)) |
| **Seeds** | 3 | stage 2 needs independent initializations to compare |
| **Bags** | one bag = one cell line, **full-line**, lines weighted equally | [Step 03](../docs/steps/03-model-and-training-design.md#mil--the-bag-model-4b_mil_training-design-fixed-13082026) |

Two runs of three seeds. `4a` must have run first.

### 2.7 · What each outcome will be written as — fixed before the numbers (Selin, 13.08.2026)

**This adds no stage, moves no bar and changes no threshold.** It fixes the *language* each outcome
receives, for the same reason §2 fixes the bars: a number does not determine a claim on its own.

§2 already did this once. A stage-7 failure means *"Q2 unanswered, the instrument was not
demonstrated"* and never *"no heterogeneity found"* — that is a decision about **wording**, taken in
advance so the result could not pick its own phrasing. The rest of the write-up never got the same
treatment, and it has the same freedom: a median cross-seed ρ of 0.03 that clears its null is
truthfully described both as *"cells are reproducibly ordered within a line"* and as *"the effect is
statistically clear but too small to matter"*. Whichever is written first becomes the finding.

| outcome | what it is written as | what it may **not** be written as |
|---|---|---|
| **stage 0 collapses** for a representation | *the representation places a line's cells at one point; no model of any kind could separate them, and Q2 is unanswerable in this encoding* | anything about the model, which never ran |
| **stage 7 fails** | *Q2 unanswered — the instrument was not demonstrated*, with the measured AUROC given | *no heterogeneity found*; the aggregator is **not** swapped and retried |
| **stage 1 fails** (MIL's spread ≤ `4a`'s) | *removing the variance penalty did not produce more within-line variation, so the architecture did not deliver the freedom it was adopted for* | *there is no heterogeneity* — this is a statement about the objective, not the biology |
| **stage 2 fails** | *the within-line variation does not reproduce across seeds; it is initialisation noise* | *the model found nothing*; stage 1 may still have passed, and that combination is itself the finding |
| **stage 2 passes but small** | the ρ **and** the null are both given in the same sentence, and the sentence says *statistically distinguishable from noise* — not *substantial*, *strong* or *clear* unless a stated comparison supports the adjective | a bare "significant" with the magnitude omitted |
| **stage 6 vetoes** | *the variation is explained by sequencing depth or cell cycle at least as well as it reproduces across seeds; it is a technical artifact that happens to be reproducible* | any softening that leaves the positive standing |
| **everything passes** | *Q2(a) is positive: per-cell predictions vary within a line, reproducibly, and not as a sequencing artifact — at a measured instrument sensitivity of AUROC x.xx* | *the model learned heterogeneity*, which is Q2(b) and is **not addressed**; nor anything about which cells survive treatment, which is Q2(c) |

**Three rules that apply to every row.**

1. **Every negative carries stage 7's AUROC in the same sentence.** A negative from an instrument of
   unknown sensitivity is not a result, and separating the two lets the sensitivity be forgotten.
2. **Magnitude adjectives require a stated comparison.** *"Small"*, *"substantial"* and *"strong"* are
   claims; each must name what it is being compared against, or the number stands unqualified.
3. **A split between representations is a result about the encoding, not a nuisance.** If `pca` and
   `scgpt` disagree at any stage, that is written as a finding about where the structure lives — not
   as one arm being reported and the other footnoted.

**What no outcome licenses.** Q2(b) — *is this real cellular heterogeneity of drug response* — and
Q2(c) — *does it identify the cells that survive treatment* — are out of reach for want of
measurements, and no result from these five stages moves them. Every write-up of a positive states
that limitation in the same paragraph as the positive, not in a later caveat.

## 3 · Running the criterion

Everything below implements §2 and **adds nothing to it**. Each stage is one cell, in the order §2
fixes — 0, 7, 1, 2, 6 — and each prints the quantity §2 says it reports, whether or not that quantity
is also a gate.

The bag model itself lives in [`scripts/training/mil.py`](../scripts/training/mil.py), not here. It is
imported for the same reason `4a` imports `scripts/training/cv.py`: the fold partition, the per-fold
density fit and the head-bias initialization must be the **same code** in both notebooks, or stage 1's
comparison rests on two implementations that agree today. `mil.py` imports `grouped_folds` and
`inner_holdout` from `cv.py` rather than re-deriving them, so fold *f* holds out the same cell lines in
`4a` and in `4b`.

In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
NB_DIR = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # runs/ lives at the project root

import anndata as ad
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, spearmanr

from scripts.layout import PipelinePaths
from scripts.training.cv import line_level_predictions
from scripts.training.density_weighting import DEFAULT_CAP, line_level
from scripts.training.mil import bag_oof_predictions
from scripts.training.training_utils import TrainConfig

# 4b's own output tree. 4a's `outputs/panel/` is the panel run's; keeping them apart is what lets a
# reader tell which architecture produced a table from its path alone. The one file that crosses
# between them is 4a's within-line spread table, which stage 1 READS and never writes.
OUT = NB_DIR / 'outputs' / 'mil'
OUT.mkdir(parents=True, exist_ok=True)
PERCELL = Path('runs') / 'percell'          # shared with 4a; the filenames carry the architecture
PERCELL.mkdir(parents=True, exist_ok=True)
PANEL_OUT = NB_DIR / 'outputs' / 'panel'

SCORE, VARIANT = 'auc_cc', 'hvg5000'
REPS = ['X_pca', 'X_scGPT']
N_SPLITS = 5

# THE PANEL IS READ, NOT RESTATED -- the same rule 4a follows, for the same reason: a literal here
# would let 4b train on a different drug set than the model it is compared against.
PANEL_CSV = PANEL_OUT / 'panel.csv'
if not PANEL_CSV.exists():
    raise FileNotFoundError(
        f'{PANEL_CSV} not found; it is written by 2_drug_selection.ipynb. There is deliberately no '
        f'fallback list.')
PANEL = pd.read_csv(PANEL_CSV)['drug_key'].tolist()

# alpha = 0.5 ONLY, and it is not swept here (§2.6). 4a sweeps {0, 0.5, 1} because the weighting is
# what that notebook is testing; here the ARCHITECTURE is the change under test, and moving two
# things at once makes the difference unattributable (docs/TODO.md, the governing rule).
ALPHA = 0.5
# Three seeds, because stage 2 asks whether independent initializations rank the same cells alike and
# two points do not make that a comparison. 42 is 4a's; 43 and 44 are its successors and are
# ARBITRARY -- any three distinct integers would do, and nothing in the data picks these.
SEEDS = (42, 43, 44)
EPOCHS = 50  # 4a's, unchanged. See the banner below for what that means at one step per line.

paths = PipelinePaths.build(None, VARIANT, SCORE)
print(f'panel        : {len(PANEL)} drugs from {PANEL_CSV.name}')
print(f'targets      : {paths.targets_h5ad.name}')
print(f'run scope    : {len(REPS)} reps x {len(SEEDS)} seeds at alpha={ALPHA} '
      f'= {len(REPS) * len(SEEDS)} runs of {N_SPLITS} folds')
print(f'weighting    : alpha={ALPHA}, cap={DEFAULT_CAP} (4a defaults; not swept here)')
print(f'epochs       : {EPOCHS} (4a\'s cap). One bag is one gradient step, so an epoch is ~120 steps '
      f'here against ~330 in 4a at batch_size=128 -- same cap, fewer updates.')

### 3.1 · Load only what is needed

Identical to `4a`'s load, deliberately: the targets h5ad is 2.4 GB and almost all of it is the
expression matrix `.X`, which neither notebook touches — both models consume the precomputed
embeddings in `obsm`. Reading it backed and assembling a light `AnnData` keeps the two runs on the
same array contents as well as the same code.

`uns['ctrp_drugs']` is rewritten to the panel because `build_bags` matches target columns against it,
exactly as `MultiDrugDataset` does.

In [ ]:
src = ad.read_h5ad(paths.targets_h5ad, backed='r')
all_drugs = list(src.uns['ctrp_drugs'])
kcol = [all_drugs.index(d) for d in PANEL]

Y_raw = np.asarray(src.obsm['Y_ctrp'], dtype=np.float32)[:, kcol]
M = np.asarray(src.obsm['M_ctrp'], dtype=bool)[:, kcol]
Y = np.where(M, Y_raw, 0.0).astype(np.float32)  # used as published: no clipping (Step 01)

adata = ad.AnnData(obs=src.obs.copy())
for rep in REPS:
    adata.obsm[rep] = np.asarray(src.obsm[rep], dtype=np.float32)
adata.obsm['Y_ctrp'], adata.obsm['M_ctrp'] = Y, M
adata.uns['ctrp_drugs'] = PANEL
src.file.close()

groups = adata.obs['Cell_line'].astype(str).to_numpy()
eligible = adata.obs['split_ctrp'].isin(['train', 'val']).to_numpy()  # fixed test set held out
lines_elig = np.unique(groups[eligible])
n_cells_per_line = pd.Series(groups[eligible]).value_counts()
print(f'{adata.n_obs} cells | {len(lines_elig)} eligible cell lines | K={len(PANEL)} drugs')
# Bag sizes are worth printing rather than assuming: one bag is one line, so this IS the distribution
# of training-example sizes, and the largest line sets peak memory (mil.py, module docstring).
print(f'bag size: min {n_cells_per_line.min()} | median {int(n_cells_per_line.median())} | '
      f'max {n_cells_per_line.max()} cells')

### 3.2 · Stage 0 — the input ceiling

The statistic §2.1 fixes: per representation, the **within-line share of total variance** — the mean
over cell lines of the within-line variance of the cell embeddings, divided by the total variance over
all eligible cells. Variance is summed over the embedding's dimensions (the trace of the covariance),
which is what makes the ratio scale-free and puts `pca` and `scgpt` on one axis despite sharing no
units. It is one minus the intraclass correlation.

**It reports, and it fails only on collapse** (§2.1 — the 0.10 floor was dropped as sourceless and
stricter than the precondition it stood for). Collapse is the precondition: cells of a line
numerically indistinguishable, so that no model of any kind could separate them. That is tested as
zero within-line variance, not as a small one.

**Which PCA.** The stored all-cells `X_pca`, which `scripts/model/dataset.py` defines as the
*descriptive* representation. This is deliberate and is the one place in the project where that is the
right key: stage 0 is a statement about the representation, made before any training, so a fold-fitted
rotation would make the answer depend on a fold assignment that plays no part in the question. The
fold-local fits (`cv.fold_pca_projections`) exist because a *model input* must not depend on held-out
cells; nothing is being fitted here.

Lines are weighted equally, as they are everywhere else in this notebook — a mean over lines, not over
cells, so a 1,990-cell line does not count 35 times a 56-cell one.

⚠️ **Timing (§2.1).** The embeddings on disk predate the preprocessing corrections and R1 re-embeds,
so a number computed before R1 is indicative, not final. The *shape* of the answer — spread versus
collapsed — is unlikely to flip.

In [ ]:
def within_line_share(X, groups, lines):
    """Mean over lines of within-line variance, over the total variance. One minus the ICC.

    Variance is the trace of the covariance -- summed over dimensions -- so the ratio is invariant to
    the scale of the embedding and `pca` and `scgpt` are comparable. ddof=1 throughout: a line's cells
    are a sample of that line, not the population.

    Returns (share, per_line) so the degree is visible and not only the ratio; §2.1 reports both.
    """
    total = float(np.var(X, axis=0, ddof=1).sum())
    per_line = pd.Series(
        {ln: float(np.var(X[groups == ln], axis=0, ddof=1).sum()) for ln in lines})
    return float(per_line.mean() / total), per_line


stage0 = []
for rep in REPS:
    X = adata.obsm[rep][eligible]
    share, per_line = within_line_share(X, groups[eligible], lines_elig)
    collapsed = per_line[per_line <= 0]
    stage0.append({'rep': rep, 'n_dims': X.shape[1], 'within_line_share': share,
                   'n_lines': len(per_line), 'n_collapsed_lines': int(collapsed.size),
                   'min_line_var': float(per_line.min()), 'median_line_var': float(per_line.median()),
                   'collapse': bool(share <= 0 or collapsed.size)})

stage0 = pd.DataFrame(stage0)
stage0.to_csv(OUT / 'stage0_input_ceiling.csv', index=False)
print(stage0.to_string(index=False))
print()
for r in stage0.itertuples():
    verdict = ('COLLAPSE -- the cells of at least one line are numerically identical; no model can '
               'separate them' if r.collapse else 'no collapse')
    print(f'{r.rep:9s}  within-line share of total variance = {r.within_line_share:.3f}  -> {verdict}')
print()
print('Reported, not gated: any value above collapse is a matter of degree and belongs in the report '
      'as a number (§2.1). A split answer between the two representations is itself a result -- it '
      'locates the structure in the encoding rather than in the model.')

### 3.3 · The runs — two representations, three seeds

Six runs of five folds. The fold partition comes from `cv.grouped_folds` and `cv.inner_holdout`, the
same deterministic functions `4a` calls, so fold *f* holds out the same cell lines in both notebooks
and under every seed — which is what makes stage 1's "same lines, same drugs, same folds" true rather
than intended.

**`counts_h5ad` is passed, and must be.** Under cross-validation a PCA representation is refitted
inside each fold on that fold's fitting cells (`cv.fold_pca_projections`, decision 2 of the three
fits): a model input may not depend on the held-out lines. The alternative — the stored all-cells
`X_pca` — is the leak that decision closed, and asking for it needs `all_cells_pca=True` said out
loud.

**`PCA_SEED = 42`, fixed across all three model seeds (Selin, 13.08.2026).** The per-fold PCA takes a
seed for its randomized SVD. `4a` passes the model seed to both, which is harmless there because it
trains one seed; here three seeds run, and the alternative was to let the representation move with
them. It is fixed because of what stage 2 measures: with the representation held constant, cross-seed
agreement isolates **model initialization**, which is how §2.4 words the stage — *do independent seeds
assign high and low predictions to the same cells*. Letting the PCA seed follow would make stage 2 a
test of the whole pipeline, stricter but unattributable: a failure could not be assigned to the model
rather than to the PCA fit, and the pass would be measured against a null that does not separate them.

It affects only the `pca` arm. scGPT estimates nothing across cells — its weights are pretrained and
frozen and its binning is per cell — so it has no fit to seed.

The cost, stated: the three seeds are then not fully independent draws of the pipeline, only of the
model. Any claim of reproducibility from stage 2 is therefore a claim about the model given this
representation, and the report says it that way.

In [ ]:
PCA_SEED = 42   # fixed across model seeds (Selin, 13.08.2026) -- see the markdown above

oof, fold_log = {}, []
for rep in REPS:
    for seed in SEEDS:
        pred, folds = bag_oof_predictions(
            adata, rep, PANEL,
            config=TrainConfig(epochs=EPOCHS, seed=seed), n_splits=N_SPLITS,
            density_weighting=ALPHA > 0, alpha=ALPHA, init_head_bias=True,
            counts_h5ad=paths.raw_h5ad, pca_seed=PCA_SEED,
            tag=f'mil_{rep}_s{seed}')
        oof[(rep, seed)] = (pred, folds)
        fold_log.extend([{**f, 'model': 'mil', 'alpha': ALPHA, 'seed': seed} for f in folds])
        print(f'== done {rep} seed={seed}')

fold_log = pd.DataFrame(fold_log)
fold_log.to_csv(OUT / 'mil_training_folds.csv', index=False)
print()
print(fold_log.groupby(['rep', 'seed'])[['best_epoch', 'best_val_obj']].mean().round(4).to_string())
print()
print('bags per fold (= training examples, = gradient steps per epoch):')
print(fold_log.groupby('fold')[['n_fit_bags', 'n_fit_cells', 'largest_bag']].first().to_string())

### 3.4 · Persisting the predictions

The per-cell prediction is this model's **native output** and every stage below reads it; the
line-level table is the mean derived from it. Both are written, in `4a`'s formats:

| Path | What | Tracked? |
|---|---|---|
| `runs/percell/percell_mil_<rep>_a0.5_s<seed>.npy` | `(n_cells, len(PANEL))` `float32`, NaN where a cell was never held out | **no** — gitignored with the rest of `runs/` |
| `outputs/mil/mil_within_line_spread.csv` | one row per (rep, seed, drug, cell line): that line's spread, cell count and mean prediction | **yes** — the table stage 1 compares against `4a`'s |
| `outputs/mil/mil_oof_predictions.csv` | the line-level tidy table, in `4a`'s column layout | **yes** — read by [`5_evaluation`](5_evaluation.ipynb) |

`cell_index.csv` and `drug_order.json` are shared with `4a` and carry the row and column identities,
so an array is never read positionally against a differently ordered `adata`. If `4a` wrote them
first, they are **checked** rather than overwritten: a disagreement means the two notebooks are not
indexing the same cells, which would make every cell-for-cell comparison below meaningless while
looking fine.

In [ ]:
index = pd.DataFrame({'cell': adata.obs_names, 'cell_line': groups})
index_csv, order_json = PERCELL / 'cell_index.csv', PERCELL / 'drug_order.json'
if index_csv.exists():
    # Written by whichever of 4a / 4b ran first. Checked, not overwritten: if the two disagree, every
    # cell-for-cell comparison below is comparing different cells and nothing downstream would say so.
    prior = pd.read_csv(index_csv)
    if not prior['cell'].equals(index['cell']):
        raise ValueError(
            f'{index_csv} lists a different cell order than this run. 4a and 4b must index the same '
            f'cells in the same order for stage 1 to compare like with like. Delete {PERCELL} and '
            f're-run both, rather than reconciling them by hand.')
    if json.loads(order_json.read_text()) != PANEL:
        raise ValueError(f'{order_json} lists a different drug order than PANEL.')
else:
    index.to_csv(index_csv, index=False)
    order_json.write_text(json.dumps(PANEL, indent=1))

spread, tidy = [], []
for (rep, seed), (pred, folds) in oof.items():
    np.save(PERCELL / f'percell_mil_{rep}_a{ALPHA:g}_s{seed}.npy', pred.astype(np.float32))
    # ddof=1: a sample spread over that line's sequenced cells, not the population it stands for.
    # The identical aggregation 4a §4 performs -- same groupby, same estimator -- because stage 1
    # compares the two columns directly and a difference in how they were computed would read as a
    # difference between the architectures.
    long = (pd.DataFrame(pred, columns=PANEL)
            .assign(cell_line=groups)
            .melt(id_vars='cell_line', var_name='drug', value_name='pred')
            .dropna(subset=['pred']))
    agg = (long.groupby(['cell_line', 'drug'], sort=False)['pred']
           .agg(n_cells='size', within_line_sd=lambda v: v.std(ddof=1), line_mean_pred='mean')
           .reset_index())
    spread.append(agg.assign(rep=rep, seed=seed))
    tidy.append(line_level_predictions(pred, adata, PANEL, folds=folds, rep=rep, model='mil',
                                       alpha=ALPHA, seed=seed))

within_mil = pd.concat(spread, ignore_index=True)[
    ['rep', 'seed', 'drug', 'cell_line', 'n_cells', 'within_line_sd', 'line_mean_pred']]
within_mil.to_csv(OUT / 'mil_within_line_spread.csv', index=False)
oof_tidy = pd.concat(tidy, ignore_index=True)
oof_tidy.to_csv(OUT / 'mil_oof_predictions.csv', index=False)

print(f'{len(oof)} runs x {pred.shape} per-cell predictions -> {PERCELL}/')
print(f'{len(within_mil)} (rep, seed, drug, line) rows -> mil_within_line_spread.csv')
print(f'{len(oof_tidy)} line-level rows -> mil_oof_predictions.csv')
print()
print('median within-line sd of per-cell predictions:')
print(within_mil.groupby(['rep', 'seed'])['within_line_sd'].median().round(5).to_string())

### 3.5 · Stage 7 — the synthetic positive control

§2.2, implemented. For each panel drug: take cell-line pairs whose measured responses differ by an
amount in the **bottom quartile** of that drug's pairwise `|y_A − y_B|`, treat the union of their cells
as one bag mixed at weight 0.5, and ask whether the model ranks the A-cells above the B-cells *inside*
that bag. The statistic is the within-bag AUROC — the Mann–Whitney U divided by `n_A · n_B` — and the
pass condition is that it beats the null obtained by permuting which cells came from which source,
which is that statistic's own null.

**No bags are materialised and no extra forward pass is made.** Instance-level MIL has no attention, so
a cell's prediction does not depend on which cells accompany it in a bag. The prediction a cell would
receive inside the synthetic bag is exactly the out-of-fold prediction already computed for it, and the
within-bag AUROC is a statistic of those two groups of numbers. This is a property of the architecture
chosen in §1, not a shortcut.

**No subsampling to equal cell counts is needed either.** AUROC is invariant to the group sizes and
Mann–Whitney's null accommodates unequal `n`; the recovered gap is a difference of two group means. The
mixture weight enters only the bag-level diagnostic, which is computed from the two group means
directly.

> ⬜ **OPEN, and Selin's — two judgement calls in this cell (13.08.2026). Neither is settled by §2.**
>
> **1 · Pairs are restricted to lines held out by the same fold (`SAME_FOLD_PAIRS = True`).** A and B
> are predicted by *different* models when they fall in different folds, so any systematic offset
> between two fold-models would enter the AUROC as if it were signal. Restricting to same-fold pairs
> removes that at the cost of roughly four fifths of the candidate pairs. The alternative is to allow
> cross-fold pairs and accept the offset. My reading, marked as a reading: restrict — a control that
> can be passed by inter-model offset is not a control.
>
> **2 · The per-pair tests are aggregated by Benjamini–Hochberg at FDR 0.05**, and the stage reports
> the fraction of pairs that survive. §2 fixes the test *per bag* and does not say how thousands of
> bags become one verdict. Alternatives: a Wilcoxon signed-rank of the per-pair AUROCs against 0.5
> (one test, but the pairs share cell lines and are not independent), or a full permutation null on
> the median AUROC (respects the dependence, costs a simulation). BH is the conventional choice and
> its 0.05 is the conventional level, not a number chosen for this data.
>
> A third point, decided rather than open: the test is **one-sided**. The direction is predicted in
> advance — cells of the more-resistant line should score higher — so a two-sided test would spend
> half its power on an outcome the criterion does not accept as a pass.

In [ ]:
SAME_FOLD_PAIRS = True   # see the OPEN note above
FDR = 0.05               # Benjamini & Hochberg, JRSS-B 57(1) 1995; the conventional level

y_lines_all, obs_lines_all = line_level(Y, M, groups, lines_elig)
y_line = pd.DataFrame(y_lines_all, index=lines_elig, columns=PANEL)
obs_line = pd.DataFrame(obs_lines_all, index=lines_elig, columns=PANEL)
line_fold = pd.Series({ln: f['fold'] for f in oof[(REPS[0], SEEDS[0])][1] for ln in f['val_lines']})
cells_of = {ln: np.flatnonzero((groups == ln) & eligible) for ln in lines_elig}


def bottom_quartile_pairs(drug):
    """Line pairs for one drug whose |y_A - y_B| falls in the bottom quartile of that drug's own
    pairwise differences, oriented so A is the more resistant line (higher label).

    The quartile is computed on ALL pairs of lines screened against the drug -- that is the drug's
    pairwise spread, and it is the reference §2.2 names. The same-fold restriction is applied
    afterwards, so it filters which of those pairs are usable and does not move the cut.
    """
    obs = obs_line[drug]
    ln = np.asarray(obs.index[obs])
    y = y_line.loc[ln, drug].to_numpy()
    i, j = np.triu_indices(len(ln), k=1)
    gap = np.abs(y[i] - y[j])
    if gap.size == 0:
        return []
    cut = np.quantile(gap, 0.25)
    keep = np.flatnonzero(gap <= cut)
    out = []
    for k in keep:
        a, b = (ln[i[k]], ln[j[k]]) if y[i[k]] >= y[j[k]] else (ln[j[k]], ln[i[k]])
        if SAME_FOLD_PAIRS and line_fold[a] != line_fold[b]:
            continue
        out.append((a, b, float(y_line.loc[a, drug] - y_line.loc[b, drug])))
    return out


PAIRS = {d: bottom_quartile_pairs(d) for d in PANEL}
print('bottom-quartile pairs per drug (after the same-fold restriction):')
print(pd.Series({d: len(v) for d, v in PAIRS.items()}).to_string())

rows = []
for (rep, seed), (pred, _) in oof.items():
    for j, drug in enumerate(PANEL):
        for a, b, gap in PAIRS[drug]:
            sa = pred[cells_of[a], j]
            sb = pred[cells_of[b], j]
            sa, sb = sa[np.isfinite(sa)], sb[np.isfinite(sb)]
            if sa.size == 0 or sb.size == 0 or gap == 0:
                continue
            # One-sided: A is the more resistant line by construction, so the predicted direction is
            # A > B. U/(n_A n_B) is the AUROC -- the probability a random A-cell outranks a random
            # B-cell -- and its p-value is the within-bag permutation null of §2.2 evaluated exactly.
            u = mannwhitneyu(sa, sb, alternative='greater')
            rows.append({'rep': rep, 'seed': seed, 'drug': drug, 'line_a': a, 'line_b': b,
                         'n_a': sa.size, 'n_b': sb.size, 'gap': gap,
                         'auroc': u.statistic / (sa.size * sb.size), 'p': u.pvalue,
                         'recovered_gap_frac': (sa.mean() - sb.mean()) / gap})

stage7 = pd.DataFrame(rows)


def bh_reject(p, q=FDR):
    """Benjamini-Hochberg step-up: boolean rejections at FDR q."""
    p = np.asarray(p, dtype=float)
    order = np.argsort(p)
    thresh = q * np.arange(1, p.size + 1) / p.size
    passed = np.flatnonzero(p[order] <= thresh)
    out = np.zeros(p.size, dtype=bool)
    if passed.size:
        out[order[:passed[-1] + 1]] = True
    return out


stage7['significant'] = False
for key, g in stage7.groupby(['rep', 'seed'], sort=False):
    stage7.loc[g.index, 'significant'] = bh_reject(g['p'])
stage7.to_csv(OUT / 'stage7_positive_control.csv', index=False)

summary7 = (stage7.groupby(['rep', 'seed'])
            .agg(n_pairs=('auroc', 'size'), median_auroc=('auroc', 'median'),
                 frac_significant=('significant', 'mean'),
                 median_recovered=('recovered_gap_frac', 'median'),
                 median_gap=('gap', 'median'))
            .round(4))
print()
print(summary7.to_string())
print()
print('AUROC is the instrument\'s MEASURED SENSITIVITY and is reported with every downstream negative '
      '(§2.2). recovered_gap_frac is reported as a description and gates nothing -- it is '
      'calibration-sensitive, and its denominator is small by construction under bottom-quartile '
      'pairing.')
print('⚠️ A stage-7 failure ENDS THE RUN and means "Q2 unanswered, the instrument was not '
      'demonstrated" -- never "no heterogeneity found". The aggregator is not swapped for one that '
      'passes (§2, and the closing note).')

### 3.6 · Stage 1 — spread, against `4a` rather than against a number

§2.3, implemented. The pass condition is that MIL's within-line standard deviation of per-cell
predictions **exceeds `4a`'s**, on the same cell lines, drugs, folds and representation — no margin,
because equation §2.3 shows the comparison to be a test of precisely the term the bag loss deletes.

The `4a` column is read from `outputs/panel/panel_within_line_spread.csv` at `alpha = 0.5`, the level
this notebook runs. **This cell raises if that file is absent**, and that is the point: stage 1 has no
fallback, so `4a` is a hard predecessor of `4b` in the R-sequence.

> ⬜ **OPEN, and Selin's — how the per-(drug, line) comparison becomes one verdict (13.08.2026).**
> §2.3 fixes *what* is compared and not how thousands of paired values are summarised. Written below
> as the **paired fraction**: the share of (drug, cell line) cells where MIL's spread exceeds `4a`'s,
> with the verdict at more than half. Alternatives: compare the two medians (simpler, but a median
> can move while most lines go the other way), or a Wilcoxon signed-rank on the paired differences
> (a p-value, but it tests the median difference rather than the direction the criterion words).
> Both are printed beside the fraction so the choice is visible rather than hidden in it.
>
> `4a` runs one seed and this notebook runs three, so the comparison is made **per MIL seed** and the
> three verdicts are printed separately. Reporting only their mean would hide a split.

In [ ]:
from scipy.stats import wilcoxon

PERCELL_4A = PANEL_OUT / 'panel_within_line_spread.csv'
if not PERCELL_4A.exists():
    raise FileNotFoundError(
        f'{PERCELL_4A} not found. Stage 1 compares this model\'s within-line spread against '
        f'4a_percell_training\'s on the same lines, drugs and folds, and there is deliberately no '
        f'fallback: 4a must run first (§2.3, the dependency note). Run 4a section A, then this cell.')

within_4a = pd.read_csv(PERCELL_4A).query('alpha == @ALPHA')
if within_4a.empty:
    raise ValueError(f'{PERCELL_4A.name} holds no rows at alpha={ALPHA}; 4a must have swept it.')

# ⚠️ NOT `pred_std` in panel_per_drug_correlation.csv, which is the spread of LINE-LEVEL predictions
# ACROSS cell lines -- a between-line quantity and the opposite of this one (§2.3).
paired = within_mil.merge(
    within_4a[['rep', 'drug', 'cell_line', 'within_line_sd', 'n_cells']],
    on=['rep', 'drug', 'cell_line'], suffixes=('_mil', '_4a'), validate='many_to_one')
if not np.array_equal(paired['n_cells_mil'], paired['n_cells_4a']):
    raise ValueError(
        'the two notebooks disagree on how many cells a (drug, cell line) contributes, so they did '
        'not hold out the same cells. Stage 1 requires matching folds (§2.3).')
print(f'{len(paired)} paired (rep, seed, drug, cell line) rows | '
      f'{paired[["drug", "cell_line"]].drop_duplicates().shape[0]} distinct (drug, line) pairs')

stage1 = (paired.assign(mil_exceeds=lambda d: d.within_line_sd_mil > d.within_line_sd_4a)
          .groupby(['rep', 'seed'])
          .apply(lambda g: pd.Series({
              'n_pairs': len(g),
              'frac_mil_exceeds': g.mil_exceeds.mean(),
              'median_sd_mil': g.within_line_sd_mil.median(),
              'median_sd_4a': g.within_line_sd_4a.median(),
              'wilcoxon_p': wilcoxon(g.within_line_sd_mil, g.within_line_sd_4a,
                                     alternative='greater').pvalue,
          }), include_groups=False)
          .round(5))
stage1['passes'] = stage1['frac_mil_exceeds'] > 0.5
stage1.to_csv(OUT / 'stage1_spread_vs_percell.csv')
print()
print(stage1.to_string())
print()
print('No margin, by construction: 4a\'s objective charges for within-line variance at full weight '
      'in every batch and the bag objective does not contain the term at all, so the bare comparison '
      'IS the test of the deleted term (§2.3).')

### 3.7 · Stage 2 — reproducibility, the actual test

§2.4, implemented. Do independent seeds assign high and low predictions to the *same* cells? For each
(representation, drug, cell line) and each of the three seed pairs, the agreement is the **Spearman
correlation across that line's cells** between the two seeds' per-cell predictions.

**The shuffled-cell null needs no simulation, because it is this statistic's own null.** §2.4 defines
the null as permuting cell identities within each line — which destroys any cell-specific
correspondence while preserving each seed's marginal distribution of predictions. That is precisely
the permutation null of a rank correlation, so `spearmanr`'s p-value evaluates it directly. Rank-based
rather than Pearson, for the same reason stage 7 is: it reads order, and order is what stages 1 and 2
rest on, while mean pooling gives the model a standing shrinkage incentive that a scale-sensitive
statistic would be sensitive to.

Lines with fewer than three cells for a drug carry no defined correlation and are counted out rather
than dropped silently.

> ⬜ **OPEN, and Selin's — the same aggregation question as stage 7 (13.08.2026).** §2.4 fixes the
> null per (drug, cell line) and does not say how ~1,700 of them become one verdict. Written below as
> Benjamini–Hochberg at FDR 0.05 with the fraction surviving reported, matching stage 7 so the two
> are read on one scale. The median ρ is printed beside it because §2.4 requires the agreement value
> to be **reported** — a result that is statistically clear but small must be visible as such rather
> than hidden behind a pass.

In [ ]:
from itertools import combinations

MIN_CELLS_FOR_RHO = 3   # a Spearman correlation is undefined below this; not a threshold on the data

rows, skipped = [], 0
for rep in REPS:
    for s1, s2 in combinations(SEEDS, 2):
        p1, p2 = oof[(rep, s1)][0], oof[(rep, s2)][0]
        for j, drug in enumerate(PANEL):
            for ln in lines_elig:
                ci = cells_of[ln]
                a, b = p1[ci, j], p2[ci, j]
                ok = np.isfinite(a) & np.isfinite(b)
                if ok.sum() < MIN_CELLS_FOR_RHO:
                    skipped += 1
                    continue
                # One-sided: agreement means the two seeds order the cells the SAME way. The p-value
                # is the within-line shuffled-cell null of §2.4, evaluated exactly rather than
                # simulated -- permuting one vector's cell identities IS this statistic's null.
                r = spearmanr(a[ok], b[ok], alternative='greater')
                rows.append({'rep': rep, 'seed_a': s1, 'seed_b': s2, 'drug': drug, 'cell_line': ln,
                             'n_cells': int(ok.sum()), 'rho': r.statistic, 'p': r.pvalue})

stage2 = pd.DataFrame(rows)
stage2['significant'] = False
for key, g in stage2.groupby(['rep', 'seed_a', 'seed_b'], sort=False):
    stage2.loc[g.index, 'significant'] = bh_reject(g['p'])
stage2.to_csv(OUT / 'stage2_cross_seed_agreement.csv', index=False)

summary2 = (stage2.groupby(['rep', 'seed_a', 'seed_b'])
            .agg(n=('rho', 'size'), median_rho=('rho', 'median'),
                 frac_significant=('significant', 'mean'))
            .round(4))
print(f'{len(stage2)} (rep, seed pair, drug, cell line) agreements | '
      f'{skipped} skipped for fewer than {MIN_CELLS_FOR_RHO} usable cells')
print()
print(summary2.to_string())
print()
print('per representation, pooled over seed pairs:')
print(stage2.groupby('rep').agg(median_rho=('rho', 'median'),
                                frac_significant=('significant', 'mean')).round(4).to_string())
print()
print('The agreement VALUE is reported, not only the verdict (§2.4): a result that is statistically '
      'clear but small must be visible as small.')

### 3.8 · Stage 6 — the confound veto

§2.5, implemented. Regress each cell's predicted response on **total counts, genes detected,
mitochondrial fraction and cell-cycle score**, within line. The veto fires when the confounds explain
the within-line variation — in which case stages 1 and 2 have measured a sequencing artifact that
happens to reproduce across seeds, because the confounds themselves reproduce across seeds.

**This is the stage that tests the strongest rival explanation for a positive result.** Stages 1 and 2
establish that per-cell predictions vary within a line and that independent seeds agree on *which*
cells. Sequencing depth produces exactly that pattern — it varies within a line and reproduces across
seeds perfectly, being a property of the cell rather than of the model. On log-CPM PCA depth is
routinely a dominant axis, so this is a live alternative and not a remote one.

> **Two covariates were unrecoverable, and were recovered (13.08.2026).** Between the criterion being
> closed and this cell being written, `total_counts` and the mitochondrial fraction turned out to be
> absent from every processed file: SCP542 reaches this project as **CPM**, so library size is divided
> out upstream of everything, and only **4** of the 13 `MT-` genes survive HVG selection. The stage
> was briefly un-evaluable, and the options were to recover the covariates, to amend a closed
> criterion down to the two that existed, or to report the veto as un-run.
>
> They were recovered, so **§2.5 runs as written and the criterion needed no amendment.** SCP542's own
> `UMIcount_data.txt` is un-normalised and carries all 13 `MT-` genes; `scripts/preprocessing/qc_covariates.py`
> reads it in one streaming pass and `scp542_conversion` writes `total_counts` and `pct_counts_mt`
> into `obs` alongside the rest of the metadata.
>
> **Both are computed over the full gene set (Selin, 13.08.2026), and that is not a precision
> argument.** A total over the highly variable genes is not depth — it is depth times the fraction of
> a cell's counts falling in that set, and *that fraction is biological*. A confound regressor
> carrying the signal under test can veto a **true** positive, which is worse than a noisy one. The
> HVG-restricted mitochondrial fraction is likewise a truncated numerator (4 of 13 genes, missing
> `CO1`, `CO2`, `CO3`, `ATP6`, `ND4` — the high expressers) over a denominator missing most of the
> transcriptome.
>
> ⚠️ **The columns arrive with the preprocessing rerun.** Any h5ad built before 13.08.2026 lacks them
> and this cell raises rather than quietly narrowing itself — a veto that silently drops covariates
> is how a blocked check turns into a passed one.

**The veto is read on the adjusted R² (Selin, 13.08.2026).** The regression carries five covariates,
and plain R² is biased upward by their number with the bias scaling as 1/n — while a cell line
contributes anywhere from 56 to 1,990 cells. The unadjusted figure is therefore inflated by a
*different amount for every line*, which is exactly the axis a single veto threshold has to be
compared across: a small line would clear a bar that a large one fails, on identical evidence. The
adjustment removes that, and the cost is that it can go negative when the confounds explain nothing,
which is a correct reading rather than a defect.

The unadjusted figure is still computed and printed, as a description. Nothing reads it.

> ⬜ **Still open, and Selin's — the magnitude at which the veto fires (13.08.2026).** §2.5 fixes
> *what* is regressed; the bar itself is now known to be on the adjusted R², but not where. **A
> permutation null cannot supply it** — with hundreds of cells in a line, an adjusted R² far too small
> to matter is still significant against one — so this is the single bar in §2 that a null cannot
> replace, and it has to be a number someone chooses. Until it is chosen the value is reported and
> **no automatic veto is issued**, which keeps the decision visible rather than buried in a constant.
> It is computed from the saved per-cell predictions, so it can be settled after the run without
> retraining anything.

In [ ]:
# The four §2.5 covariates and the obs column each resolves to. total_counts and pct_counts_mt are
# written by scripts/preprocessing/qc_covariates.py from the raw UMI matrix; Genes_expressed and the
# cell-cycle scores come from SCP542's own metadata. Cell cycle is TWO columns, G1 and G2, and both
# enter the regression -- collapsing them to one score would be a choice nobody made.
CONFOUNDS = {
    'total_counts': 'total_counts',
    'genes_detected': 'Genes_expressed',
    'mito_fraction': 'pct_counts_mt',
    # ⚠️ The columns are 'G1/S_score' and 'G2/M_score', NOT 'G1'/'G2' (fixed 13.08.2026, Gate 5).
    # The earlier names were read off h5py GROUP keys: a forward slash in a column name becomes a
    # group hierarchy in the h5 file, so `obs/G1` is a group containing `S_score` and looks like a
    # column called 'G1' to anything inspecting the file structure instead of the column-order
    # attribute. Both would have raised KeyError on the first real run.
    'cell_cycle_G1S': 'G1/S_score',
    'cell_cycle_G2M': 'G2/M_score',
}
missing = [f'{k} -> obs[{v!r}]' for k, v in CONFOUNDS.items() if v not in adata.obs.columns]
if missing:
    raise KeyError(
        'the confound veto is defined on covariates this h5ad does not carry: '
        + '; '.join(missing) + '. total_counts and pct_counts_mt are written by '
        'scripts/preprocessing/qc_covariates.py from SCP542\'s raw UMIcount_data.txt, and any file '
        'built before 13.08.2026 predates them -- re-run the convert step. This raises rather than '
        'dropping the covariate, because a veto that silently narrows itself turns a blocked check '
        'into a passed one (§3.8).')

C = adata.obs[list(CONFOUNDS.values())].to_numpy(dtype=float)


def within_line_r2(y, Xc):
    """(R^2, adjusted R^2) of the per-cell predictions on the covariates, centred within the line.

    Centring within line is what makes this a statement about WITHIN-line variation: the between-line
    difference is exactly what stages 1 and 2 are not about, and leaving it in would let a covariate
    that merely differs between lines look like an explanation.

    BOTH are returned because the choice between them changes the number a veto bar would be set on,
    and that bar is Selin's (§3.8). Plain R^2 is biased upward by the number of regressors -- five
    here -- and the bias scales with 1/n, while a cell line contributes anywhere from 56 to 1,990
    cells. So the plain figure is inflated by a different amount for every line, which is exactly the
    axis a single threshold is compared across. The adjusted figure removes that, at the cost of
    being able to go negative when the covariates explain nothing.
    """
    n, p = Xc.shape
    y = y - y.mean()
    Xc = Xc - Xc.mean(0)
    denom = float((y ** 2).sum())
    if denom <= 0 or n <= p + 1:
        return np.nan, np.nan
    beta, *_ = np.linalg.lstsq(Xc, y, rcond=None)
    r2 = 1.0 - float(((y - Xc @ beta) ** 2).sum()) / denom
    return r2, 1.0 - (1.0 - r2) * (n - 1) / (n - p - 1)


rows = []
for (rep, seed), (pred, _) in oof.items():
    for j, drug in enumerate(PANEL):
        for ln in lines_elig:
            ci = cells_of[ln]
            y = pred[ci, j]
            ok = np.isfinite(y) & np.isfinite(C[ci]).all(1)
            if ok.sum() < len(CONFOUNDS) + 2:
                continue
            r2, r2_adj = within_line_r2(y[ok], C[ci][ok])
            rec = {'rep': rep, 'seed': seed, 'drug': drug, 'cell_line': ln,
                   'n_cells': int(ok.sum()), 'r2_confounds': r2, 'r2_confounds_adj': r2_adj}
            for name, col in CONFOUNDS.items():
                rec[f'rho_{name}'] = spearmanr(y[ok], adata.obs[col].to_numpy(float)[ci][ok]).statistic
            rows.append(rec)

stage6 = pd.DataFrame(rows)
stage6.to_csv(OUT / 'stage6_confounds.csv', index=False)
# VETO_STAT is the column the veto is read on (Selin, 13.08.2026): the ADJUSTED R^2, because the
# unadjusted one is inflated by the five regressors by an amount that scales with 1/n, and a line
# contributes 56 to 1,990 cells -- so a single bar on the unadjusted figure would be a different bar
# for every line. Named once here rather than spelled out at each use, so the choice is one edit.
VETO_STAT = 'r2_confounds_adj'

cols = [VETO_STAT, 'r2_confounds'] + [f'rho_{n}' for n in CONFOUNDS]
print(f'covariates: {dict(CONFOUNDS)}')
print(f'veto is read on: {VETO_STAT} (adjusted for {len(CONFOUNDS)} regressors)')
print()
print('median over (drug, cell line), per run:')
print(stage6.groupby(['rep', 'seed'])[cols].median().round(4).to_string())
print()
print('distribution of the within-line variance explained by the confounds:')
print(stage6.groupby('rep')[[VETO_STAT, 'r2_confounds']]
      .describe()[[(VETO_STAT, '50%'), (VETO_STAT, '75%'), (VETO_STAT, 'max'),
                   ('r2_confounds', '50%'), ('r2_confounds', 'max')]].round(4).to_string())
print()
print(f'{VETO_STAT} is the veto quantity; r2_confounds is printed as a description and nothing reads '
      'it. The adjusted figure can go negative -- that reads as "the confounds explain nothing", '
      'which is the outcome this stage hopes for.')
print()
print('⬜ NO AUTOMATIC VETO YET: the bar is known to be on the adjusted R^2 but its MAGNITUDE is not '
      'set, and a permutation null cannot set it -- with hundreds of cells per line an adjusted R^2 '
      'far too small to matter is still significant. It is the one bar in §2 a null cannot replace, '
      'so the value is reported and the decision stays visible (§3.8).')

### 3.9 · The verdict

Assembled from the five stages in §2's own order, and it says only what §2 licenses it to say. Nothing
below computes a new quantity; if a statement here is not traceable to a stage above, it does not
belong.

Two rules from §2 that this cell enforces rather than restates:

- **A stage-7 failure ends the run**, and means *Q2 unanswered, the instrument was not demonstrated* —
  never *no heterogeneity found*. Stages 1, 2 and 6 are not consulted, because a negative from an
  instrument that was never shown to work is uninterpretable.
- **Every negative is reported with the instrument's measured sensitivity attached** — stage 7's
  AUROC — so a reader can tell a demonstrated absence from a blunt instrument.

In [ ]:
verdict = []
for rep in REPS:
    s0 = stage0.set_index('rep').loc[rep]
    s7 = summary7.loc[rep]
    s1 = stage1.loc[rep]
    s2 = summary2.loc[rep]
    row = {
        'rep': rep,
        'stage0_within_line_share': round(float(s0.within_line_share), 4),
        'stage0_collapse': bool(s0.collapse),
        'stage7_median_auroc': round(float(s7.median_auroc.median()), 4),
        'stage7_frac_significant': round(float(s7.frac_significant.median()), 4),
        'stage1_frac_mil_exceeds': round(float(s1.frac_mil_exceeds.median()), 4),
        'stage1_passes_all_seeds': bool(s1.passes.all()),
        'stage2_median_rho': round(float(s2.median_rho.median()), 4),
        'stage2_frac_significant': round(float(s2.frac_significant.median()), 4),
        # Evaluated now that the UMI covariates exist (13.08.2026). What is still open is the
        # magnitude at which the veto FIRES, so the R^2 is carried here and read, not gated.
        'stage6_median_r2': round(float(stage6.query('rep == @rep')['r2_confounds'].median()), 4),
        'stage6_median_r2_adj': round(float(stage6.query('rep == @rep')['r2_confounds_adj'].median()), 4),
    }
    verdict.append(row)

verdict = pd.DataFrame(verdict)
verdict.to_csv(OUT / 'q2_verdict.csv', index=False)
print(verdict.to_string(index=False))
print()
for r in verdict.itertuples():
    print(f'--- {r.rep} ---')
    if r.stage0_collapse:
        print('  stage 0 COLLAPSE: the cells of a line are numerically identical in this '
              'representation. No model can separate them; later stages measure nothing.')
        continue
    print(f'  stage 0: within-line share {r.stage0_within_line_share:.3f} (reported, not gated)')
    print(f'  stage 7: median within-bag AUROC {r.stage7_median_auroc:.3f}, '
          f'{r.stage7_frac_significant:.1%} of pairs beat the permutation null')
    if r.stage7_frac_significant <= FDR:
        print('  ⛔ STAGE 7 FAILED -> THE RUN ENDS HERE. Q2 UNANSWERED, THE INSTRUMENT WAS NOT '
              'DEMONSTRATED. This is NOT "no heterogeneity found", and the aggregator is not '
              'swapped for one that passes (§2, closing note).')
        continue
    print(f'  stage 1: MIL spread exceeds 4a\'s on {r.stage1_frac_mil_exceeds:.1%} of (drug, line) '
          f'pairs; all three seeds pass: {r.stage1_passes_all_seeds}')
    print(f'  stage 2: median cross-seed rho {r.stage2_median_rho:.3f}, '
          f'{r.stage2_frac_significant:.1%} beat the shuffled-cell null')
    print(f'  stage 6: adjusted R^2 on the four confounds, median '
          f'{r.stage6_median_r2_adj:.4f} (unadjusted {r.stage6_median_r2:.4f}) -- the veto reads the\n'
          f'           ADJUSTED figure (§3.8). Its magnitude is still open, so this does not\n'
          f'           automatically overturn the verdict.')
    positive = (r.stage1_passes_all_seeds and r.stage2_frac_significant > FDR)
    print(f'  => Q2(a) {"POSITIVE" if positive else "NEGATIVE"} for {r.rep}, at a measured '
          f'instrument sensitivity of AUROC {r.stage7_median_auroc:.3f}.')
    print('     Q2(b) -- is this real heterogeneity of drug response -- and Q2(c) -- does it predict '
          'which cells survive -- are NOT addressed and cannot be with these measurements (§1).')

## 4 · Closing analysis — what kind of cells were they?

Short and descriptive, and **it gates nothing**. By the time it runs, §2 has already decided whether Q2
is positive. This section says *what was found*, not *whether* something was found — the distinction
matters, because an enrichment discovered here cannot be promoted into evidence afterwards.

Two figures and a table:

**a · What the predictions track (stage 6, reported rather than vetoing).** The same regression the veto
uses, shown rather than thresholded: how much of the within-line variation in per-cell predictions is
explained by each of total counts, genes detected, mitochondrial fraction and cell-cycle score. If the
veto passed, these are all small, and showing them is what makes that credible.

**b · Which annotated programs the predictions track (stage 3).** Kinker et al. 2020 annotated recurrent
heterogeneity programs for this exact dataset, independently of any drug-response label. Both sides are
continuous — each cell has a program score, and instance-level MIL gives each cell a predicted response —
so **correlate the two across a line's cells**, per program, against a within-line permutation null.

*No top-k, decided 12.08.2026 (Selin).* An earlier draft took the most- and least-resistant predicted
cells and tested them for enrichment. Correlating the full continuous signal is strictly better here: it
needs no `k` to justify, uses every cell instead of a slice, and is the same method as (a) above — so the
confound check and the biology check are read on one scale rather than two.

⚠️ **Read the enrichment carefully.** Cell-cycle enrichment is close to guaranteed and would be weak
evidence of anything: the project already refuted *the cell-line effect is largely proliferation*
([Corrections](../docs/steps/corrections-and-dead-ends.md#the-cell-line-effect-is-largely-proliferation)),
and Kinker's two named associations are recorded as **not transferring to this task**
([Dead ends](../docs/steps/corrections-and-dead-ends.md#kinkers-two-named-associations-do-not-transfer-to-this-task)).
A program *other* than cell cycle would be the interesting outcome.

**c · One table** — per drug: does the cell ordering repeat across drugs, or is it drug-specific? A
general axis and a drug-specific subpopulation are different findings, and the table is the cheapest
way to tell them apart.

### 4.1 · Implementing the closing analysis

**(a) is §3.8's table, shown rather than thresholded** — it is already printed there, and is not
recomputed here. Note what that means given §3.8's finding: the part of (a) that would make the veto
credible is the part that cannot currently be computed.

**(b) and (c) are below.** Both are descriptive and **gate nothing**: §2 has already decided whether
Q2 is positive by the time this runs, and an association discovered here cannot be promoted into
evidence afterwards. The permutation nulls exist so that "the predictions track program X" is not
asserted from a correlation that any two continuous within-line signals would produce.

For (b), the null is again the within-line shuffled-cell null and again is evaluated exactly rather
than simulated — permuting cells within a line is the permutation null of the rank correlation. Cell
cycle is included among the programs deliberately, so that the ⚠️ in §4 is visible in the output
rather than only in the prose: cell-cycle association is close to guaranteed and would be weak
evidence of anything, and a program **other** than cell cycle is the interesting outcome.

In [ ]:
# Kinker's programs and the two cell-cycle scores all carry the '_score' suffix, so one
# comprehension collects them. It used to append ['G1', 'G2'] as well, which named no real column
# and double-counted the cell-cycle scores it was trying to add (fixed 13.08.2026, Gate 5).
PROGRAMS = [c for c in adata.obs.columns if c.endswith('_score')]
print(f'{len(PROGRAMS)} annotated programs: {PROGRAMS}')

rows = []
for (rep, seed), (pred, _) in oof.items():
    for j, drug in enumerate(PANEL):
        for prog in PROGRAMS:
            s = adata.obs[prog].to_numpy(dtype=float)
            for ln in lines_elig:
                ci = cells_of[ln]
                y, x = pred[ci, j], s[ci]
                ok = np.isfinite(y) & np.isfinite(x)
                if ok.sum() < MIN_CELLS_FOR_RHO:
                    continue
                r = spearmanr(y[ok], x[ok])
                rows.append({'rep': rep, 'seed': seed, 'drug': drug, 'program': prog,
                             'cell_line': ln, 'n_cells': int(ok.sum()),
                             'rho': r.statistic, 'p': r.pvalue})

programs = pd.DataFrame(rows)
programs['significant'] = False
for key, g in programs.groupby(['rep', 'seed'], sort=False):
    programs.loc[g.index, 'significant'] = bh_reject(g['p'])
programs.to_csv(OUT / 'closing_program_correlations.csv', index=False)

print()
print('median within-line rho between per-cell prediction and program score, pooled over drugs, '
      'seeds and lines:')
print(programs.groupby(['rep', 'program'])
      .agg(median_rho=('rho', 'median'), frac_significant=('significant', 'mean'))
      .round(3).unstack(0).to_string())
print()
print('⚠️ Cell-cycle association (G1, G2) is close to guaranteed and is weak evidence of anything: '
      'this project already refuted "the cell-line effect is largely proliferation", and Kinker\'s '
      'two named associations are on record as not transferring to this task (§4). A program other '
      'than cell cycle is the interesting outcome.')

In [ ]:
# (c) Does the cell ordering repeat across drugs, or is it drug-specific? A general axis -- "these
# cells score high for everything" -- and a drug-specific subpopulation are different findings, and
# this table is the cheapest way to tell them apart. Within line, so it is a statement about the
# ordering of cells and not about the lines' differing sensitivities.
rows = []
for (rep, seed), (pred, _) in oof.items():
    for j1, j2 in combinations(range(len(PANEL)), 2):
        for ln in lines_elig:
            ci = cells_of[ln]
            a, b = pred[ci, j1], pred[ci, j2]
            ok = np.isfinite(a) & np.isfinite(b)
            if ok.sum() < MIN_CELLS_FOR_RHO:
                continue
            rows.append({'rep': rep, 'seed': seed, 'drug_a': PANEL[j1], 'drug_b': PANEL[j2],
                         'cell_line': ln, 'n_cells': int(ok.sum()),
                         'rho': spearmanr(a[ok], b[ok]).statistic})

cross_drug = pd.DataFrame(rows)
cross_drug.to_csv(OUT / 'closing_cross_drug_ordering.csv', index=False)
print('median within-line rho between the per-cell orderings of two drugs:')
print(cross_drug.groupby('rep')['rho'].describe()[['count', '25%', '50%', '75%']].round(3).to_string())
print()
print('per drug, against all others:')
both = pd.concat([cross_drug.rename(columns={'drug_a': 'drug', 'drug_b': 'other'}),
                  cross_drug.rename(columns={'drug_b': 'drug', 'drug_a': 'other'})])
print(both.groupby(['rep', 'drug'])['rho'].median().round(3).unstack(0).reindex(PANEL).to_string())
print()
print('High and uniform -> one general axis, and the per-drug heads are reading the same thing. '
      'Low -> drug-specific orderings. Descriptive: this gates nothing (§4).')

### The criterion is closed (13.08.2026)

**No stage may be added to §2, and none may be dropped from it, once a run exists.** Building the model
first and choosing the criterion afterwards is the failure mode this notebook was written backwards to
prevent — which is why §2 was fixed in prose before any cell below it existed.

That applies to the aggregator in particular. If stage 7 fails, mean pooling is **not** swapped for
something that passes: *fail → change the instrument → retry* is a forking path moved down one level,
and it would leave any subsequent positive unattributable. A stage-7 failure ends the run and means
**Q2 unanswered, the instrument was not demonstrated** — never "no heterogeneity found".